In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Funciones para trabajo con Colecciones").getOrCreate()

In [2]:
data = spark.read.parquet("./data/parquet/")

A veces una columna particular en un conjunto de datos contiene una lista de valores una forma de modelar eso es mediante los tipos de datos Arrays

In [4]:
data.show(truncate=False)

+-----+--------------------------------------------+
|dia  |tareas                                      |
+-----+--------------------------------------------+
|lunes|[hacer la tarea, buscar agua, lavar el auto]|
+-----+--------------------------------------------+



In [5]:
data.printSchema()

root
 |-- dia: string (nullable = true)
 |-- tareas: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [6]:
from pyspark.sql.functions import col, size, sort_array, array_contains

## funcion size

In [7]:
data.select(
    size(col("tareas")).alias("tamaño"),
    sort_array(col("tareas")).alias("Array_ordenado"),
    array_contains(col("tareas"), "buscar agua").alias("buscar_agua")
).show(truncate=False)

+------+--------------------------------------------+-----------+
|tamaño|Array_ordenado                              |buscar_agua|
+------+--------------------------------------------+-----------+
|3     |[buscar agua, hacer la tarea, lavar el auto]|true       |
+------+--------------------------------------------+-----------+



In [8]:
from pyspark.sql.functions import explode

In [9]:
data.select(
    col("dia"),
    explode(col("tareas")).alias("tarea")
).show(truncate=False)

+-----+--------------+
|dia  |tarea         |
+-----+--------------+
|lunes|hacer la tarea|
|lunes|buscar agua   |
|lunes|lavar el auto |
+-----+--------------+



In [10]:
# esta manteniendo el dia de la semana y por cada dia lo que esta haciendo es construyendo tantas filas como tareas existen en nuestro array

## Formato JSON

In [12]:
json_df_str = spark.read.parquet("./data/JSON")

In [13]:
json_df_str.show(truncate=False)


+---------------------------------------------------------------------------+
|tareas_str                                                                 |
+---------------------------------------------------------------------------+
|{"dia": "lunes","tareas": ["hacer la tarea","buscar agua","lavar el auto"]}|
+---------------------------------------------------------------------------+



In [14]:
# si se fijan tiene un formato JSON de tipo string

json_df_str.printSchema()

root
 |-- tareas_str: string (nullable = true)



In [16]:
# necesitamos describir su estructura

from pyspark.sql.types import StructType, StringType, ArrayType, StructField

schema_json = StructType([
    StructField("dia", StringType(), True),
    StructField("tareas", ArrayType(StringType()), True)
])

In [18]:
from pyspark.sql.functions import from_json, to_json

In [19]:
json_df = json_df_str.select(
    from_json(col("tareas_str"), schema_json).alias("tareas_por_hacer")
)

In [20]:
json_df.printSchema()

root
 |-- tareas_por_hacer: struct (nullable = true)
 |    |-- dia: string (nullable = true)
 |    |-- tareas: array (nullable = true)
 |    |    |-- element: string (containsNull = true)



In [22]:
json_df.select(
    col("tareas_por_hacer").getItem("dia"),
    col("tareas_por_hacer").getItem("tareas"),
    col("tareas_por_hacer").getItem("tareas").getItem(0).alias("primer_tarea")
).show(truncate=False)

+--------------------+--------------------------------------------+--------------+
|tareas_por_hacer.dia|tareas_por_hacer.tareas                     |primer_tarea  |
+--------------------+--------------------------------------------+--------------+
|lunes               |[hacer la tarea, buscar agua, lavar el auto]|hacer la tarea|
+--------------------+--------------------------------------------+--------------+



## Inversamente lo que seria la funcion JSON con to_json

In [24]:
json_df.select(
    to_json(col("tareas_por_hacer"))
).show(truncate=False)

+-------------------------------------------------------------------------+
|to_json(tareas_por_hacer)                                                |
+-------------------------------------------------------------------------+
|{"dia":"lunes","tareas":["hacer la tarea","buscar agua","lavar el auto"]}|
+-------------------------------------------------------------------------+

